# 01 数据探索与预处理

> 北京交通大学《机器学习与Python编程》研究性专题 — 裂纹图像识别系统

本 Notebook 包含：
1. 数据集概览与可视化
2. 图像预处理方法对比（CLAHE、高斯滤波、中值滤波）
3. 特征提取方法演示（HOG、LBP、GLCM、边缘密度）

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

# 添加项目根目录到 sys.path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_ROOT
from src.data_utils import (
    apply_clahe,
    apply_gaussian_filter,
    apply_median_filter,
    extract_edge_density,
    extract_glcm_features,
    extract_hog_features,
    extract_lbp_features,
    load_dataset,
    split_dataset,
)
from src.plot_config import set_chinese_font

set_chinese_font()

## 1. 数据集概览

加载完整数据集（正负样本各 20000 张），统计基本信息并可视化展示。

In [ ]:
# 加载完整数据集
from tqdm.auto import tqdm
import time

print("正在加载完整数据集，请稍候...")
t0 = time.time()
images, labels = load_dataset(DATA_ROOT)
elapsed = time.time() - t0

print(f"数据集总量：{len(images)} 张")
print(f"正样本（有裂纹）：{np.sum(labels == 1)} 张")
print(f"负样本（无裂纹）：{np.sum(labels == 0)} 张")
print(f"图像尺寸：{images[0].shape}")
print(f"数据类型：{images.dtype}")
print(f"内存占用：约 {images.nbytes / 1024**3:.1f} GB")
print(f"加载耗时：{elapsed:.1f} 秒")

pos_indices = np.where(labels == 1)[0]
neg_indices = np.where(labels == 0)[0]

In [ ]:
# 随机展示部分样本（正负各 5 张）
rng = np.random.default_rng(42)
pos_show = rng.choice(pos_indices, 5, replace=False)
neg_show = rng.choice(neg_indices, 5, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, idx in enumerate(pos_show):
    axes[0, i].imshow(images[idx], cmap="gray")
    axes[0, i].set_title("有裂纹")
    axes[0, i].axis("off")
for i, idx in enumerate(neg_show):
    axes[1, i].imshow(images[idx], cmap="gray")
    axes[1, i].set_title("无裂纹")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

### 1.1 全局统计信息

对比正负样本的平均亮度和灰度分布差异。

In [ ]:
# 全局统计：每张图片的平均亮度
pos_means = np.array([images[i].mean() for i in pos_indices])
neg_means = np.array([images[i].mean() for i in neg_indices])

print(f"有裂纹图像 平均亮度: {pos_means.mean():.1f} ± {pos_means.std():.1f}")
print(f"无裂纹图像 平均亮度: {neg_means.mean():.1f} ± {neg_means.std():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 平均亮度分布
axes[0].hist(pos_means, bins=50, alpha=0.7, label="有裂纹", color="tomato", density=True)
axes[0].hist(neg_means, bins=50, alpha=0.7, label="无裂纹", color="steelblue", density=True)
axes[0].set_xlabel("平均亮度")
axes[0].set_ylabel("密度")
axes[0].set_title("正负样本平均亮度分布")
axes[0].legend()

# 全局灰度直方图（随机采样 1000 张）
sample_idx = rng.choice(len(images), 1000, replace=False)
pos_gray = np.concatenate([images[i].ravel() for i in sample_idx if labels[i] == 1])
neg_gray = np.concatenate([images[i].ravel() for i in sample_idx if labels[i] == 0])
axes[1].hist(pos_gray, bins=256, range=[0, 256], alpha=0.6, label="有裂纹", color="tomato", density=True)
axes[1].hist(neg_gray, bins=256, range=[0, 256], alpha=0.6, label="无裂纹", color="steelblue", density=True)
axes[1].set_xlabel("灰度值")
axes[1].set_ylabel("密度")
axes[1].set_title("正负样本灰度分布（采样 1000 张）")
axes[1].legend()

plt.tight_layout()
plt.show()

### 1.2 数据集划分

按 70% / 15% / 15% 的比例划分为训练集、验证集和测试集，采用分层抽样保证类别比例一致。

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(
    images, labels, train_ratio=0.7, val_ratio=0.15, random_seed=42
)

print(f"训练集: {len(y_train)} 张 (正: {np.sum(y_train==1)}, 负: {np.sum(y_train==0)})")
print(f"验证集: {len(y_val)} 张 (正: {np.sum(y_val==1)}, 负: {np.sum(y_val==0)})")
print(f"测试集: {len(y_test)} 张 (正: {np.sum(y_test==1)}, 负: {np.sum(y_test==0)})")

# 可视化划分比例
split_names = ["训练集", "验证集", "测试集"]
split_counts = [len(y_train), len(y_val), len(y_test)]
split_pos = [int(np.sum(y_train == 1)), int(np.sum(y_val == 1)), int(np.sum(y_test == 1))]
split_neg = [int(np.sum(y_train == 0)), int(np.sum(y_val == 0)), int(np.sum(y_test == 0))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(split_names))
w = 0.35
axes[0].bar(x - w/2, split_pos, w, label="有裂纹", color="tomato")
axes[0].bar(x + w/2, split_neg, w, label="无裂纹", color="steelblue")
axes[0].set_xticks(x)
axes[0].set_xticklabels(split_names)
axes[0].set_ylabel("样本数")
axes[0].set_title("各子集正负样本数量")
axes[0].legend()

axes[1].pie(split_counts, labels=[f"{n}\n({c}张)" for n, c in zip(split_names, split_counts)],
            autopct="%1.1f%%", colors=["#4e79a7", "#f28e2b", "#59a14f"], startangle=90)
axes[1].set_title("数据集划分比例")

plt.tight_layout()
plt.show()

## 2. 图像预处理方法对比

In [ ]:
# 选取一张有裂纹的图像作为示例
sample_img = images[pos_indices[0]]
print(f"示例图像尺寸：{sample_img.shape}，数据类型：{sample_img.dtype}")

### 2.1 CLAHE 自适应直方图均衡化

CLAHE 将图像分块进行直方图均衡化，能增强裂缝与背景的对比度。

In [ ]:
# CLAHE 不同参数对比
clahe_results = [
    ("原图", sample_img),
    ("CLAHE (clip=2.0)", apply_clahe(sample_img, clip_limit=2.0)),
    ("CLAHE (clip=3.0)", apply_clahe(sample_img, clip_limit=3.0)),
    ("CLAHE (clip=4.0)", apply_clahe(sample_img, clip_limit=4.0)),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, (title, img) in enumerate(clahe_results):
    axes[0, i].imshow(img, cmap="gray")
    axes[0, i].set_title(title)
    axes[0, i].axis("off")
    axes[1, i].hist(img.ravel(), bins=256, range=[0, 256], color="steelblue")
    axes[1, i].set_title(f"{title} 直方图")
    axes[1, i].set_xlabel("灰度值")
    axes[1, i].set_ylabel("频数")
plt.tight_layout()
plt.show()

### 2.2 高斯滤波

高斯滤波通过加权平均去除高斯噪声，保留图像的整体结构。

In [ ]:
# 高斯滤波不同核大小对比
gaussian_results = [
    ("原图", sample_img),
    ("高斯 (3×3)", apply_gaussian_filter(sample_img, kernel_size=(3, 3))),
    ("高斯 (5×5)", apply_gaussian_filter(sample_img, kernel_size=(5, 5))),
    ("高斯 (7×7)", apply_gaussian_filter(sample_img, kernel_size=(7, 7))),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (title, img) in enumerate(gaussian_results):
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(title)
    axes[i].axis("off")
plt.tight_layout()
plt.show()

### 2.3 中值滤波

中值滤波用邻域中值替代中心值，对椒盐噪声有良好效果。

In [ ]:
# 中值滤波不同核大小对比
median_results = [
    ("原图", sample_img),
    ("中值 (k=3)", apply_median_filter(sample_img, kernel_size=3)),
    ("中值 (k=5)", apply_median_filter(sample_img, kernel_size=5)),
    ("中值 (k=7)", apply_median_filter(sample_img, kernel_size=7)),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (title, img) in enumerate(median_results):
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(title)
    axes[i].axis("off")
plt.tight_layout()
plt.show()

### 2.4 预处理方法组合效果

将 CLAHE 增强与滤波去噪组合，观察最终效果。

In [ ]:
# 组合：CLAHE + 高斯滤波
clahe_gauss = apply_gaussian_filter(apply_clahe(sample_img), kernel_size=(3, 3))
# 组合：CLAHE + 中值滤波
clahe_median = apply_median_filter(apply_clahe(sample_img), kernel_size=3)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(sample_img, cmap="gray")
axes[0].set_title("原图")
axes[0].axis("off")
axes[1].imshow(clahe_gauss, cmap="gray")
axes[1].set_title("CLAHE + 高斯(3x3)")
axes[1].axis("off")
axes[2].imshow(clahe_median, cmap="gray")
axes[2].set_title("CLAHE + 中值(k=3)")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## 3. 特征提取方法演示

### 3.1 HOG 特征（方向梯度直方图）

HOG 捕获图像梯度方向分布，对裂缝的方向性特征敏感。

In [ ]:
# HOG 特征提取与可视化
from skimage.feature import hog as skimage_hog

hog_features, hog_image = skimage_hog(
    sample_img,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    visualize=True,
)
print(f"HOG 特征维度：{hog_features.shape}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(sample_img, cmap="gray")
axes[0].set_title("原图")
axes[0].axis("off")
axes[1].imshow(hog_image, cmap="hot")
axes[1].set_title("HOG 可视化")
axes[1].axis("off")
plt.tight_layout()
plt.show()

### 3.2 LBP 特征（局部二值模式）

LBP 捕获局部纹理模式，对路面微观纹理变化敏感。

In [ ]:
from skimage.feature import local_binary_pattern

radius = 1
n_points = 8
lbp_img = local_binary_pattern(sample_img, n_points, radius, method="uniform")
lbp_features = extract_lbp_features(sample_img, radius=radius, n_points=n_points)
print(f"LBP 特征维度：{lbp_features.shape}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(sample_img, cmap="gray")
axes[0].set_title("原图")
axes[0].axis("off")
axes[1].imshow(lbp_img, cmap="gray")
axes[1].set_title("LBP 图像")
axes[1].axis("off")
n_bins = n_points * (n_points - 1) + 3
axes[2].bar(range(n_bins), lbp_features, color="steelblue")
axes[2].set_title("LBP 直方图特征")
axes[2].set_xlabel("Bin")
axes[2].set_ylabel("密度")
plt.tight_layout()
plt.show()

### 3.3 GLCM 特征（灰度共生矩阵）

GLCM 刻画像素灰度级的空间共生关系，提取 contrast、correlation、energy、homogeneity 四个统计量。

In [ ]:
glcm_features = extract_glcm_features(sample_img, distances=(1, 3, 5))
print(f"GLCM 特征维度：{glcm_features.shape}")
print(f"特征向量：{glcm_features}")

# 按属性分组展示
prop_names = ["Contrast", "Correlation", "Energy", "Homogeneity"]
n_distances = 3
n_angles = 4
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, name in enumerate(prop_names):
    values = glcm_features[i::4]  # 每4个取一个属性
    axes[i].bar(range(len(values)), values, color="steelblue")
    axes[i].set_title(name)
    axes[i].set_xlabel("(距离, 方向) 组合")
    axes[i].set_ylabel("值")
plt.tight_layout()
plt.show()

### 3.4 边缘密度

裂缝区域经 Canny 边缘检测后，边缘像素密度显著高于正常路面。

In [ ]:
edge_density = extract_edge_density(sample_img)
edges = cv2.Canny(sample_img, 50, 150)
print(f"边缘密度：{edge_density:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(sample_img, cmap="gray")
axes[0].set_title("原图")
axes[0].axis("off")
axes[1].imshow(edges, cmap="gray")
axes[1].set_title(f"Canny 边缘 (密度={edge_density:.4f})")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 4. 裂缝 vs 非裂缝图像特征对比

In [ ]:
# 批量特征对比：从正负样本中各采样 500 张计算统计量
N_SAMPLE = 500
batch_pos = rng.choice(pos_indices, N_SAMPLE, replace=False)
batch_neg = rng.choice(neg_indices, N_SAMPLE, replace=False)

# 计算边缘密度
ed_pos_all = [extract_edge_density(images[i]) for i in batch_pos]
ed_neg_all = [extract_edge_density(images[i]) for i in batch_neg]

# 计算 GLCM 特征（energy 和 contrast）
glcm_pos_e, glcm_pos_c = [], []
glcm_neg_e, glcm_neg_c = [], []
for idx in batch_pos:
    g = extract_glcm_features(images[idx], distances=(1,))
    glcm_pos_e.append(g[2])  # energy
    glcm_pos_c.append(g[0])  # contrast
for idx in batch_neg:
    g = extract_glcm_features(images[idx], distances=(1,))
    glcm_neg_e.append(g[2])
    glcm_neg_c.append(g[0])

ed_pos_all = np.array(ed_pos_all)
ed_neg_all = np.array(ed_neg_all)
glcm_pos_e, glcm_pos_c = np.array(glcm_pos_e), np.array(glcm_pos_c)
glcm_neg_e, glcm_neg_c = np.array(glcm_neg_e), np.array(glcm_neg_c)

print(f"=== 批量特征对比（正负各 {N_SAMPLE} 张采样）===")
print(f"{'特征':<20} {'有裂纹 (均值±标准差)':<30} {'无裂纹 (均值±标准差)':<30}")
print("-" * 80)
print(f"{'边缘密度':<20} {ed_pos_all.mean():.4f} ± {ed_pos_all.std():.4f}       {ed_neg_all.mean():.4f} ± {ed_neg_all.std():.4f}")
print(f"{'GLCM能量':<20} {glcm_pos_e.mean():.4f} ± {glcm_pos_e.std():.4f}       {glcm_neg_e.mean():.4f} ± {glcm_neg_e.std():.4f}")
print(f"{'GLCM对比度':<20} {glcm_pos_c.mean():.2f} ± {glcm_pos_c.std():.2f}    {glcm_neg_c.mean():.2f} ± {glcm_neg_c.std():.2f}")

# 可视化特征分布
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(ed_pos_all, bins=40, alpha=0.7, label="有裂纹", color="tomato", density=True)
axes[0].hist(ed_neg_all, bins=40, alpha=0.7, label="无裂纹", color="steelblue", density=True)
axes[0].set_xlabel("边缘密度")
axes[0].set_ylabel("密度")
axes[0].set_title("边缘密度分布")
axes[0].legend()

axes[1].hist(glcm_pos_e, bins=40, alpha=0.7, label="有裂纹", color="tomato", density=True)
axes[1].hist(glcm_neg_e, bins=40, alpha=0.7, label="无裂纹", color="steelblue", density=True)
axes[1].set_xlabel("GLCM Energy")
axes[1].set_title("GLCM 能量分布")
axes[1].legend()

axes[2].hist(glcm_pos_c, bins=40, alpha=0.7, label="有裂纹", color="tomato", density=True)
axes[2].hist(glcm_neg_c, bins=40, alpha=0.7, label="无裂纹", color="steelblue", density=True)
axes[2].set_xlabel("GLCM Contrast")
axes[2].set_title("GLCM 对比度分布")
axes[2].legend()

plt.tight_layout()
plt.show() #展示图像

## 5. 小结

### 数据探索
- 完整数据集 40000 张（正负各 20000），图像尺寸统一为 227×227
- 已按 70%/15%/15% 分层划分为训练集/验证集/测试集

### 图像预处理
- **CLAHE** 能有效增强裂缝与背景的对比度，适合作为预处理步骤
- **高斯/中值滤波** 可去除噪声，但过大的核尺寸会模糊裂缝细节
- **组合使用** CLAHE + 轻度滤波可以在增强对比度的同时去噪

### 特征提取
- **HOG** 捕获裂缝的方向性梯度特征，可视化能清晰看到裂缝线条
- **LBP** 捕获局部纹理模式，区分路面材质差异
- **GLCM** 提供纹理统计量，批量统计验证裂缝区域的对比度更高、能量更低
- **边缘密度** 是最直观且区分度最高的特征，批量统计显示裂缝区域边缘密度约为正常路面的 3-4 倍